# Hybrid Machine Learning Model - Voting Classifier
## Combining Decision Tree & Random Forest for Phishing Detection

This notebook creates a **hybrid model** using a **Voting Classifier** that combines:
1. **Decision Tree Classifier** (from Phishing_DT.ipynb)
2. **Random Forest Classifier** (from Phishing_Random_Forests.ipynb)

The model is trained using **all 41 features** from the dataset to maximise detection accuracy
and reduce false positives (legitimate URLs incorrectly flagged as phishing).

**Voting Strategies:**
- **Hard Voting** – Each model votes for a class; the majority wins.
- **Soft Voting** – Each model outputs class probabilities; the class with the highest average probability wins.

## Step 1: Install & Import Libraries

In [1]:
import sys
!{sys.executable} -m pip install pandas
!{sys.executable} -m pip install matplotlib
!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install scikit-learn

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score,
    matthews_corrcoef, ConfusionMatrixDisplay, roc_curve, auc
)
import joblib
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load & Preprocess the Dataset

In [3]:
# Load dataset (same dataset used in both original notebooks)
df_phishing_csv = pd.read_csv('datasets/Dataset.csv', on_bad_lines='skip')
print(f"Original dataset shape: {df_phishing_csv.shape}")
print(f"Has NaN values: {df_phishing_csv.isnull().values.any()}")

Original dataset shape: (116600, 26)
Has NaN values: True


In [4]:
# Clean dataset - remove any rows with missing values
cleaned_df = df_phishing_csv.dropna(how='any')
print(f"Cleaned dataset shape: {cleaned_df.shape}")
print(f"\nClass distribution (0=Legitimate, 1=Phishing):")
print(cleaned_df['Type'].value_counts())

Cleaned dataset shape: (116586, 26)

Class distribution (0=Legitimate, 1=Phishing):


KeyError: 'Type'

In [ ]:
# Preview dataset
cleaned_df.head()

In [ ]:
cleaned_df.describe()

In [ ]:
# Display all feature names (columns) in the dataset
feature_names = list(cleaned_df.columns[1:])  # All columns except 'Type'
print(f"Total number of features: {len(feature_names)}")
print(f"\nAll features used for training:")
for i, feat in enumerate(feature_names, 1):
    print(f"  {i:2d}. {feat}")

In [ ]:
cleaned_df.info()

In [ ]:
# Check for missing values
print(cleaned_df.isnull().sum())

In [ ]:
# Distribution of target variable
print(cleaned_df['Type'].value_counts())

## Step 3: Feature Selection (All 41 Features) & Train-Test Split

Using **all 41 features** from the dataset to give the hybrid model the richest possible signal for accurate phishing detection.

In [ ]:
# Feature selection - use ALL features (same as original DT and RF notebooks)
X = cleaned_df.iloc[:, 1:].values  # All features except the first column (Type)
y = cleaned_df.iloc[:, 0].values   # Target variable (Type: 0=Legitimate, 1=Phishing)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
print(X)

In [ ]:
print(y)

In [ ]:
# Split dataset into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
print(y_test)

In [ ]:
# Feature scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

## Step 4: Build Individual Models (Base Estimators)
Building the same models used in the original notebooks with all 41 features, so we can compare them to the hybrid.

In [ ]:
# Model 1: Decision Tree Classifier (same as Phishing_DT.ipynb)
dt_clf = DecisionTreeClassifier(random_state=0)
dt_clf.fit(X_train, y_train)
y_pred_dt = dt_clf.predict(X_test)

print("=" * 60)
print("MODEL 1: DECISION TREE CLASSIFIER (All 41 Features)")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.6f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.6f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.6f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt):.6f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_pred_dt):.6f}")
print(f"MCC:       {matthews_corrcoef(y_test, y_pred_dt):.6f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Confusion Matrix for Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)
print("Confusion Matrix (Decision Tree):")
print(cm_dt)

In [ ]:
# Model 2: Random Forest Classifier (same as Phishing_Random_Forests.ipynb)
rf_clf = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

print("=" * 60)
print("MODEL 2: RANDOM FOREST CLASSIFIER (All 41 Features)")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.6f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.6f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.6f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.6f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_pred_rf):.6f}")
print(f"MCC:       {matthews_corrcoef(y_test, y_pred_rf):.6f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion Matrix for Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix (Random Forest):")
print(cm_rf)

## Step 5: Build the Hybrid Voting Classifier

The **VotingClassifier** combines both models using two strategies:
- **Hard Voting**: Each classifier votes for a class label; the majority vote wins.
- **Soft Voting**: Each classifier outputs probability estimates; the class with the highest average probability wins.

In [ ]:
# ==========================================
# HYBRID MODEL: HARD VOTING CLASSIFIER
# ==========================================
# Each model casts a vote, and the majority class wins

hard_voting_clf = VotingClassifier(
    estimators=[
        ('decision_tree', DecisionTreeClassifier(random_state=0)),
        ('random_forest', RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0))
    ],
    voting='hard'
)

hard_voting_clf.fit(X_train, y_train)
y_pred_hard = hard_voting_clf.predict(X_test)

print("=" * 60)
print("HYBRID MODEL: HARD VOTING CLASSIFIER (All 41 Features)")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_hard):.6f}")
print(f"Precision: {precision_score(y_test, y_pred_hard):.6f}")
print(f"Recall:    {recall_score(y_test, y_pred_hard):.6f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_hard):.6f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_pred_hard):.6f}")
print(f"MCC:       {matthews_corrcoef(y_test, y_pred_hard):.6f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_hard))

In [ ]:
# Confusion Matrix for Hard Voting
cm_hard = confusion_matrix(y_test, y_pred_hard)
print("Confusion Matrix (Hard Voting):")
print(cm_hard)

In [ ]:
# ==========================================
# HYBRID MODEL: SOFT VOTING CLASSIFIER
# ==========================================
# Each model outputs class probabilities; the class with the highest average probability wins

soft_voting_clf = VotingClassifier(
    estimators=[
        ('decision_tree', DecisionTreeClassifier(random_state=0)),
        ('random_forest', RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0))
    ],
    voting='soft'
)

soft_voting_clf.fit(X_train, y_train)
y_pred_soft = soft_voting_clf.predict(X_test)

print("=" * 60)
print("HYBRID MODEL: SOFT VOTING CLASSIFIER (All 41 Features)")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_soft):.6f}")
print(f"Precision: {precision_score(y_test, y_pred_soft):.6f}")
print(f"Recall:    {recall_score(y_test, y_pred_soft):.6f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_soft):.6f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_pred_soft):.6f}")
print(f"MCC:       {matthews_corrcoef(y_test, y_pred_soft):.6f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_soft))

In [ ]:
# Confusion Matrix for Soft Voting
cm_soft = confusion_matrix(y_test, y_pred_soft)
print("Confusion Matrix (Soft Voting):")
print(cm_soft)

## Step 6: Comparative Analysis - All Models

In [ ]:
# Create a comprehensive comparison DataFrame
results = {
    'Model': ['Decision Tree', 'Random Forest', 'Hybrid (Hard Voting)', 'Hybrid (Soft Voting)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_hard),
        accuracy_score(y_test, y_pred_soft)
    ],
    'Precision': [
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_hard),
        precision_score(y_test, y_pred_soft)
    ],
    'Recall': [
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_hard),
        recall_score(y_test, y_pred_soft)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_hard),
        f1_score(y_test, y_pred_soft)
    ],
    'ROC AUC': [
        roc_auc_score(y_test, y_pred_dt),
        roc_auc_score(y_test, y_pred_rf),
        roc_auc_score(y_test, y_pred_hard),
        roc_auc_score(y_test, y_pred_soft)
    ],
    'MCC': [
        matthews_corrcoef(y_test, y_pred_dt),
        matthews_corrcoef(y_test, y_pred_rf),
        matthews_corrcoef(y_test, y_pred_hard),
        matthews_corrcoef(y_test, y_pred_soft)
    ]
}

results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
results_df = results_df.round(6)
print("\n" + "=" * 90)
print("COMPREHENSIVE MODEL COMPARISON (All 41 Features)")
print("=" * 90)
results_df

In [ ]:
# Highlight the best model for each metric
print("\nBest model per metric:")
print("-" * 40)
for col in results_df.columns:
    best_model = results_df[col].idxmax()
    best_value = results_df[col].max()
    print(f"{col:12s}: {best_model} ({best_value:.6f})")

## Step 7: Visualisations

In [ ]:
# Bar chart comparison of all models across all metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Performance Comparison: Individual vs Hybrid Voting (All 41 Features)', fontsize=16, fontweight='bold')

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC', 'MCC']
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
model_names = ['Decision Tree', 'Random Forest', 'Hybrid\n(Hard Voting)', 'Hybrid\n(Soft Voting)']

for idx, metric in enumerate(metrics_list):
    ax = axes[idx // 3, idx % 3]
    values = results_df[metric].values
    bars = ax.bar(model_names, values, color=colors, edgecolor='black', linewidth=0.5)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(min(values) - 0.05, max(values) + 0.03)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices side by side
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Confusion Matrices Comparison (All 41 Features)', fontsize=16, fontweight='bold')

cms = [cm_dt, cm_rf, cm_hard, cm_soft]
titles = ['Decision Tree', 'Random Forest', 'Hybrid (Hard Voting)', 'Hybrid (Soft Voting)']

for ax, cm, title in zip(axes, cms, titles):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Phishing'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

# Get probability scores for ROC curves
dt_proba = dt_clf.predict_proba(X_test)[:, 1]
rf_proba = rf_clf.predict_proba(X_test)[:, 1]
soft_proba = soft_voting_clf.predict_proba(X_test)[:, 1]

for name, proba, color in [
    ('Decision Tree', dt_proba, '#e74c3c'),
    ('Random Forest', rf_proba, '#2ecc71'),
    ('Hybrid (Soft Voting)', soft_proba, '#9b59b6')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Individual Models vs Hybrid Voting (All 41 Features)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Grouped bar chart - all metrics in one plot
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(metrics_list))
width = 0.2

for i, (model, color) in enumerate(zip(results_df.index, colors)):
    values = results_df.loc[model].values
    bars = ax.bar(x + i * width, values, width, label=model, color=color, edgecolor='black', linewidth=0.5)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('All Models - Performance Across All Metrics (All 41 Features)', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_list, fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0.85, 1.0)
plt.tight_layout()
plt.savefig('all_metrics_grouped.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Save Models & Feature Names for Django App

In [ ]:
# Save the hybrid models and scaler for use in the Django application

joblib.dump(hard_voting_clf, 'phishing_hybrid_voting_hard.pkl')
joblib.dump(soft_voting_clf, 'phishing_hybrid_voting_soft.pkl')
joblib.dump(sc, 'standard_scaler.pkl')

# IMPORTANT: Save the feature names so the Django app knows the exact order
joblib.dump(feature_names, 'feature_names.pkl')

print("Models and artifacts saved successfully:")
print("  - phishing_hybrid_voting_hard.pkl  (Hard Voting model)")
print("  - phishing_hybrid_voting_soft.pkl   (Soft Voting model)")
print("  - standard_scaler.pkl               (StandardScaler fitted on training data)")
print("  - feature_names.pkl                 (List of 41 feature names in correct order)")
print(f"\nFeature order saved ({len(feature_names)} features):")
for i, feat in enumerate(feature_names, 1):
    print(f"  {i:2d}. {feat}")

## Step 9: Summary & Conclusion

In [ ]:
print("=" * 70)
print("SUMMARY: HYBRID VOTING CLASSIFIER RESULTS (All 41 Features)")
print("=" * 70)
print("\nModels Combined:")
print("  1. Decision Tree Classifier")
print("  2. Random Forest Classifier (n_estimators=10, criterion=entropy)")
print(f"\nTotal Features Used: {len(feature_names)}")
print("\nVoting Strategies Tested:")
print("  - Hard Voting: Majority vote from both classifiers")
print("  - Soft Voting: Average probability from both classifiers")
print("\n" + "-" * 70)
print("\nFinal Comparison Table:")
print(results_df.to_string())
print("\n" + "-" * 70)

# Determine best overall model
avg_scores = results_df.mean(axis=1)
best_overall = avg_scores.idxmax()
print(f"\nBest overall model (by average score): {best_overall}")
print(f"Average score across all metrics: {avg_scores[best_overall]:.6f}")

print("\n" + "=" * 70)
print("KEY INSIGHT:")
print("Using all 41 features provides the model with the richest possible")
print("signal for classification. This reduces false positives (legitimate")
print("URLs flagged as phishing) by giving the model more contextual")
print("information to distinguish between edge cases.")
print("=" * 70)

---
### THE END